# Stress perturbations: visual check (P2.4)

One image at every severity of every perturbation in `configs/experiments/stress.yaml`. Rows are perturbations; column 0 is the clean image. Uses the first MVTec AD hazelnut test image under `ARGUS_DATA_DIR` if it is there, otherwise a synthetic pattern.

In [ ]:
from PIL import Image, ImageDraw

from argus.stress.perturbations import PERTURBATIONS, load_levels, perturb
from argus.utils.config import data_dir

found = sorted((data_dir() / "mvtec_ad" / "hazelnut" / "test").glob("*/*.png"))
if found:
    img = Image.open(found[0]).convert("RGB")
else:  # synthetic: gradient plus fine stripes, so blur and JPEG artefacts show
    img = Image.linear_gradient("L").resize((256, 256)).convert("RGB")
    draw = ImageDraw.Draw(img)
    for x in range(0, 256, 6):
        draw.line([(x, 96), (x, 160)], fill=(220, 60, 40), width=2)
img.thumbnail((256, 256))
print(found[0] if found else "synthetic image")

In [ ]:
levels = load_levels()
w, h = img.size
cols = 1 + max(len(v) for v in levels.values())
grid = Image.new("RGB", (cols * w, len(PERTURBATIONS) * (h + 20)), "white")
draw = ImageDraw.Draw(grid)
for r, name in enumerate(PERTURBATIONS):
    for s in range(len(levels[name]) + 1):
        x, y = s * w, r * (h + 20)
        grid.paste(perturb(img, name, s, levels), (x, y + 20))
        label = "clean" if s == 0 else f"{name} {s}: {levels[name][s - 1]:g}"
        draw.text((x + 4, y + 4), label, fill="black")
grid